# Chapter 5 — Hallucination Energy

**Book alignment:** Hallucination From First Principles, Chapter 5

**Question this notebook isolates:** Does the reference `hallucination_energy` (SVD projection, NO_EVIDENCE vs MEASURED state) score supported synthetic claims low and off-topic claims high, with rank-sweep monotonicity?

Synthetic embeddings below demonstrate the geometry type only and do not reproduce the book's empirical runs.

In [ ]:
import numpy as np

SEED = 5
rng = np.random.default_rng(SEED)
print("numpy", np.__version__, "seed", SEED)

## 1 — Reference implementation with evidence state

The chapter's contract distinguishes `NO_EVIDENCE` (no reference to measure against) from `MEASURED` (projection residual). We implement that reference exactly, including the book's 2-D worked example.

In [ ]:
def hallucination_energy(claim_vec, evidence_vecs, rank_r=8):
    c = np.asarray(claim_vec, dtype=np.float32)
    E = np.asarray(evidence_vecs, dtype=np.float32)
    if c.ndim != 1 or c.size == 0:
        raise ValueError("claim_vec must be a non-empty 1D vector")
    if E.size == 0:
        return {"state": "NO_EVIDENCE", "energy": 1.0, "explained": 0.0, "effective_rank": 0}
    if E.ndim == 1:
        E = E.reshape(1, -1)
    if E.ndim != 2:
        raise ValueError("evidence_vecs must be a 2D matrix")
    if E.shape[1] != c.shape[0]:
        raise ValueError("claim/evidence embedding dimensions differ")
    c = c / max(np.linalg.norm(c), 1e-12)
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    norms = np.where(norms < 1e-12, 1.0, norms)
    E = E / norms
    _, s, Vt = np.linalg.svd(E, full_matrices=False)
    r = min(rank_r, Vt.shape[0])
    basis = Vt[:r].T
    coords = basis.T @ c
    explained = float(coords @ coords)
    energy = float(np.clip(1.0 - explained, 0.0, 1.0))
    return {"state": "MEASURED", "energy": energy, "explained": explained,
            "effective_rank": int(np.sum(s > 1e-6))}


# Book Section 9 worked example: evidence spans x-axis, claim (0.8, 0.6).
two_d = hallucination_energy([0.8, 0.6], [[1.0, 0.0]], rank_r=1)
empty = hallucination_energy([0.8, 0.6], np.zeros((0, 2)), rank_r=1)
print("2-D energy:", two_d)
print("empty-evidence record:", empty)

In [ ]:
assert two_d["state"] == "MEASURED"
assert abs(two_d["energy"] - 0.36) < 1e-6
assert abs(two_d["explained"] - 0.64) < 1e-6
assert empty["state"] == "NO_EVIDENCE" and empty["energy"] == 1.0
assert 0.0 <= two_d["energy"] <= 1.0
print("2-D residual identity X+H=1 holds; NO_EVIDENCE kept distinct from MEASURED.")

## 2 — Supported claims score low, off-topic claims score high

Evidence spans a low-rank subspace of a 64-D space. Supported claims are noisy linear combinations of evidence rows (in-span); off-topic claims are independent random directions (mostly out-of-span).

In [ ]:
D, N, RANK = 64, 10, 8
E = rng.standard_normal((N, D)).astype(np.float32)
n_each = 50
W = rng.standard_normal((n_each, N)).astype(np.float32)
supported = W @ E + 0.01 * rng.standard_normal((n_each, D)).astype(np.float32)
offtopic = rng.standard_normal((n_each, D)).astype(np.float32)
e_sup = np.array([hallucination_energy(c, E, RANK)["energy"] for c in supported])
e_off = np.array([hallucination_energy(c, E, RANK)["energy"] for c in offtopic])
print(f"supported mean={e_sup.mean():.4f} max={e_sup.max():.4f}")
print(f"off-topic mean={e_off.mean():.4f} min={e_off.min():.4f}")
print(f"mean gap={float(e_off.mean() - e_sup.mean()):.4f}")

In [ ]:
assert e_sup.mean() < 0.25, e_sup.mean()
assert e_off.mean() > 0.60, e_off.mean()
assert (e_off.mean() - e_sup.mean()) > 0.30
assert bool(np.all((e_sup >= 0.0) & (e_sup <= 1.0)))
assert bool(np.all((e_off >= 0.0) & (e_off <= 1.0)))
print("Containment separates in-span from off-topic synthetic claims.")

## 3 — Rank is a capacity control: energy falls monotonically

For a fixed nested SVD basis, explained mass can only grow with retained rank, so `H(r+1) <= H(r)`. We sweep rank on frozen vectors.

In [ ]:
ranks = [1, 2, 4, 8, 10]
demo_sup = supported[0]
demo_off = offtopic[0]
row_sup = [hallucination_energy(demo_sup, E, r)["energy"] for r in ranks]
row_off = [hallucination_energy(demo_off, E, r)["energy"] for r in ranks]
mean_sup = [float(np.mean([hallucination_energy(c, E, r)["energy"] for c in supported])) for r in ranks]
mean_off = [float(np.mean([hallucination_energy(c, E, r)["energy"] for c in offtopic])) for r in ranks]
print("rank:", ranks)
print("demo supported:", [round(v, 4) for v in row_sup])
print("demo off-topic:", [round(v, 4) for v in row_off])
print("mean supported:", [round(v, 4) for v in mean_sup])
print("mean off-topic:", [round(v, 4) for v in mean_off])

In [ ]:
for seq in (row_sup, row_off, mean_sup, mean_off):
    assert all(b <= a + 1e-6 for a, b in zip(seq, seq[1:])), seq
assert row_sup[-1] < 0.05 and row_off[-1] < row_off[0]
print("Monotonicity H(r+1)<=H(r) holds; full evidence rank drives in-span energy near zero.")

## What we earned

The reference sensor works as contracted: the 2-D identity holds, missing evidence stays a distinct state, in-span synthetic claims score low while off-topic claims score high, and rank controls capacity monotonically.

Chapter 6 keeps this sensor fixed and asks the evaluation question: what does a good score distribution still fail to guarantee at a strict operating point?